# Figure 2

In [1]:
options(warn=-1)

In [147]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(
        
            # Seurat 
            library(Seurat), 
            library(SeuratWrappers), 
                         
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ggplot2), 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel), 
            library(circlize), 
            library(ggridges), 
            library(paletteer), 
            library(egg), 
            library(ggh4x), 
            
            # Parallel 
            library(BiocParallel), 

            # Pyhton compatibility
            library(reticulate)

        )
    )
)

In [3]:
# Configure reticulate 
use_condaenv(condaenv="p.3.10.16-FD20250213HSCT", conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)

In [4]:
random_seed <- 42
set.seed(random_seed)

In [5]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [6]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=4)) # From project global source()
# showtext::showtext_auto() 

In [7]:
# Source scripts 
source("bin/seurat_qc.R")
source("script/figure/bin/dp_feature.R")
source("script/figure/bin/line_plot.R")
source("script/figure/bin/dea_vp.R")
source("script/figure/bin/dea_stats.R")

# Import Seurat object and subset for T cells / NK

In [8]:
so <- readRDS("data/object/pp.rds")
so <- NormalizeData(so, assay="RNA", normalization.method="LogNormalize", scale.factor=1e4)

Normalizing layer: counts



In [9]:
sc <- import("scanpy")

adata <- sc$read_h5ad("data/object/scvi/subset/adata_0.h5ad")
umap <- adata$obsm["X_umap"]
colnames(umap) <- c("UMAP_1", "UMAP_2")
rownames(umap) <- adata$obs_names$to_list()

In [10]:
so <- so[, rownames(umap)]
so[["umap"]] <- CreateDimReducObject(embeddings=umap, key="UMAP_", assay="RNA")

In [11]:
# Ensure levels are set to subset
so@meta.data <- droplevels(so@meta.data)
color$celltype_low_r <- color$celltype_low_r[names(color$celltype_low_r) %in% levels(so$celltype_low)]

# UMAP

In [12]:
dplot_1 <- dplot(so, reduction="umap", group_by="celltype_low", alpha=1, pt_size=0.5, size_select=4) +

    ggtitle("Cell type leiden") +
    scale_color_manual(values=color$celltype_low_r, na.value="grey50", labels=base::str2expression(levels(so$celltype_low))) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm")) + 
    theme(
    
        legend.text.align=0
    
) 

In [13]:
pdf("result/figures/figure_2/umap_celltype_low.pdf", width=10, height=3)

gridExtra::grid.arrange(
    
    dplot_1 %>% egg::set_panel_size(., width=unit(5, "cm"), height=unit(5, "cm"))

)

dev.off()

pdf 
  2

In [14]:
fplot_1 <- fplot(so, reduction="umap", features="clone_id_size", alpha=1, pt_size=0.5, size_select=4, viridis_option="A") + 
    ggtitle("Clonal expansion") + guides(color=guide_colourbar(title="#Clones", barwidth=unit(2, "mm"), barheight=unit(10, "mm"))) 

In [15]:
pdf("result/figures/figure_2/umap_clone_id_size.pdf", width=10, height=3)

gridExtra::grid.arrange(
    
    fplot_1 %>% egg::set_panel_size(., width=unit(5, "cm"), height=unit(5, "cm"))

)

dev.off()

pdf 
  2

# Marker gene expression 

In [16]:
# Genes and categories
genes <- read.csv("script/figure/figure_2/data/figure_2_dotplot_celltype_markers_fine_t_cells.csv")

In [17]:
dp_1 <- dp_feature(so, genes$gene, group_by="celltype_low", group_by_order=rev(levels(so$celltype_low)), split=NA, split_order=unique(genes$group), range_max=3, scale=TRUE) + 
    scale_y_discrete(labels=base::str2expression(rev(levels(so$celltype_low)))) 

`summarise()` has grouped output by 'celltype_low'. You can override using the
`.groups` argument.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


In [18]:
pdf("result/figures/figure_2/dp_celltype_markers_fine_t_cells.pdf", width=nrow(genes)*0.1+2.5, height=1.5+length(unique(genes$group))*0.1)

set_height <- dp_1$data$celltype_low %>% unique() %>% length() * 2.5
set_width <- dp_1$data %>% dplyr::select(gene, split) %>% dplyr::distinct() %>% dplyr::group_by(split) %>% dplyr::summarise(n=n()) %>% dplyr::pull(n) * 2.5

gridExtra::grid.arrange(
    
    dp_1 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), rows=unit(set_height, "mm"))

)

dev.off()

pdf 
  2

## Skin T cells 

In [19]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_high/results_blood.rds")

In [20]:
pdf("result/figures/figure_2/hm_dea_counts_time_point_celltype_high_blood.pdf", width=5, height=5)

mat <- dea_res_fetch(results, unique(so_3$celltype_high))
mat <- log10(mat+1)
draw(dea_res_hm(mat, fontsize_select=2))

dev.off()

pdf 
  2

In [21]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_high/results_skin.rds")

In [22]:
pdf("result/figures/figure_2/hm_dea_counts_time_point_celltype_high_skin.pdf", width=5, height=5)

mat <- dea_res_fetch(results, unique(so_3$celltype_high))
mat <- log10(mat+1)
draw(dea_res_hm(mat, fontsize_select=2))

dev.off()

pdf 
  2

In [23]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_blood.rds")

In [24]:
pdf("result/figures/figure_2/hm_dea_counts_time_point_celltype_low_blood.pdf", width=5, height=5)

mat <- dea_res_fetch(results, unique(so_3$celltype_high))
mat <- log10(mat+1)
draw(dea_res_hm(mat, fontsize_select=2))

dev.off()

pdf 
  2

In [25]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_skin.rds")

In [26]:
pdf("result/figures/figure_2/hm_dea_counts_time_point_celltype_low_skin.pdf", width=5, height=5)

mat <- dea_res_fetch(results, unique(so_3$celltype_high))
mat <- log10(mat+1)
draw(dea_res_hm(mat, fontsize_select=2))

dev.off()

pdf 
  2

# DEA Baseline vs GVHD selected genes

In [27]:
genes <- read.csv("script/figure/figure_2/data/figure_2_tcells_skin_heatmap_deg_baseline_vs_gvhd.csv")

In [28]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "celltype_low"
cell_min <- 1

In [29]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Skin")

In [30]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [31]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [32]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [33]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [34]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [35]:
# Subset by cell type 
cnt <- cnt[, grepl("RM", colnames(cnt))]

In [36]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [37]:
# Column order by time points
mat <- cnt[, order(factor(sapply(strsplit(colnames(cnt), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [38]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$celltype_low_r))
column_split <- droplevels(column_split)

In [39]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {

    # Subset samples
    mat <- mat[, column_split==levels(column_split)[idx]]
    # mat <- mat[rowSums(mat>=0)>=1, ]

    # Normalize 
    mat <- edgeR::DGEList(mat)
    mat <- edgeR::calcNormFactors(mat, method="TMM")
    mat <- edgeR::cpm(mat, normalized.lib.sizes=TRUE, log=TRUE, prior.count=1)

    # Subset genes
    mat <- mat[genes$gene, ]

    # Scale
    mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

    # Color function - DEG count
    color_ramp_mat <- c(rev(RColorBrewer::brewer.pal(11,"RdBu"))[1], "#ffffff", rev(RColorBrewer::brewer.pal(11,"RdBu"))[11])
    breaks_mat <- c(-2, 0, 2)
    color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            cluster_row_slices=FALSE,
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [40]:
pdf("result/figures/figure_2/hm_t_cell_skin_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=10)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

pdf 
  2

## Blood T cells 

In [41]:
genes <- read.csv("script/figure/figure_2/data/figure_2_tcells_blood_heatmap_deg_baseline_vs_gvhd.csv")

In [42]:
grouping_var <- "time_point"
random_effect_var <- "patient_id"
cell_type <- "celltype_low"
cell_min <- 1

In [43]:
# Subset Seurat object
so_i <- subset(so, hto_demux_class=="Blood")

In [44]:
# Get Counts 
cnt <- GetAssayData(so_i, assay="RNA", slot="counts")

In [45]:
# Prepare count data for split 
cnt <- t(as.matrix(cnt))
cnt <- as.data.frame(cnt)

In [46]:
# Make pseudobulks by suming single cell counts
cnt <- split(cnt, f=paste0(so_i[[grouping_var, drop=TRUE]], "|", so_i[[random_effect_var, drop=TRUE]], "|", so_i[[cell_type, drop=TRUE]]))

In [47]:
# Filter by min cell count 
cnt <- cnt[sapply(cnt, function(x) nrow(x)>=cell_min)]

In [48]:
cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
cnt <- do.call(cbind, cnt)

In [49]:
# Subset by cell type 
cnt <- cnt[, grepl("N/CM|E/EM", colnames(cnt))]

In [50]:
# Subset by time point
cnt <- cnt[, grepl("Baseline|GVHD", colnames(cnt))]

In [51]:
# Column order by time points
mat <- cnt[, order(factor(sapply(strsplit(colnames(cnt), "\\|"), `[[`, 1), levels=names(color$time_point)))]

In [52]:
# Column split
column_split <- sapply(strsplit(colnames(mat), "\\|"), `[[`, 3) 
column_split <- factor(column_split, levels=names(color$celltype_low_r))
column_split <- droplevels(column_split)

In [53]:
hm_list <- lapply(seq_along(levels(column_split)), function(idx) {

    # Subset samples
    mat <- mat[, column_split==levels(column_split)[idx]]
    # mat <- mat[rowSums(mat>=3)>=3, ]

    # Normalize 
    mat <- edgeR::DGEList(mat)
    mat <- edgeR::calcNormFactors(mat, method="TMM")
    mat <- edgeR::cpm(mat, normalized.lib.sizes=TRUE, log=TRUE, prior.count=1)

    # Subset genes
    mat <- mat[genes$gene, ]

    # Scale
    mat <- t(scale(t(mat), center=TRUE, scale=TRUE))

    # Color function - DEG count
    color_ramp_mat <- c(rev(brewer.pal(11,"RdBu"))[1], "#ffffff", rev(brewer.pal(11,"RdBu"))[11])
    breaks_mat <- c(-2, 0, 2)
    color_function_mat <- circlize::colorRamp2(breaks_mat, color_ramp_mat) 
    
    # Column split 
    column_split_idx <- factor(sapply(strsplit(colnames(mat), "\\|"), `[[`, 1), levels=names(color$time_point))
    
    # Get row annotation 
    top_annotation <- ComplexHeatmap::HeatmapAnnotation(

        time_point=column_split_idx %>% droplevels(), 


            annotation_legend_param=list(

                time_point=list(title="Time point", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), grid_height=unit(2, "mm"), legend_width=unit(2, "mm"), legend_height=unit(2, "mm"), border="asis")
                
            ), 

            col=list(

                time_point=color$time_point
            ), 

            simple_anno_size=unit(2.5, "mm"), 
            show_annotation_name=FALSE, 
            show_legend=TRUE

    )
    
    hm <- ComplexHeatmap::Heatmap(
    
            matrix=mat,

            col=color_function_mat, 
            na_col="white", 

            width=unit(1.25*ncol(mat), "mm"), 
            height=unit(2*nrow(mat), "mm"), 

            row_title=NULL, 
            row_title_gp=gpar(fontsize=8, fontface="bold"), 

            column_title=parse(text=levels(column_split)[idx]), 
            column_title_gp=gpar(fontsize=8, fontface="bold"), 

            row_names_gp=gpar(fontsize=6, fontface="italic"), 
            column_names_gp=gpar(fontsize=8, fontface="plain"), 

            cluster_rows=FALSE, 
            cluster_row_slices=FALSE,
            show_row_dend=FALSE,
            row_split=genes$regulation, 
            row_gap=unit(2, "mm"), 
            show_row_names=TRUE, 
            row_names_side="left", 

            cluster_columns=TRUE, 
            cluster_column_slices=FALSE, 
            show_column_dend=FALSE, 
            column_split=column_split_idx, 
            column_gap=unit(1, "mm"), 
            show_column_names=FALSE, 

            top_annotation=top_annotation, 
            left_annotation=NULL, 

            rect_gp=gpar(col=NA, lwd=0, alpha=1), 

            heatmap_legend_param=list(title="z-score", title_gp=gpar(fontsize=6, fontface="plain"), labels_gp=gpar(fontsize=6), grid_width=unit(2, "mm"), legend_width=unit(2, "mm"), grid_height=unit(10, "mm"), legend_height=unit(10, "mm"), at=breaks_mat), 
            show_heatmap_legend=ifelse(idx==1, TRUE, FALSE),

            border=TRUE, 
            border_gp=gpar(col="black", lwd=unit(0.2, "mm")), 

            use_raster=FALSE, raster_by_magick=FALSE

    )

}
                 )

hm <- Reduce(`+`, hm_list)

In [54]:
pdf("result/figures/figure_2/hm_t_cell_blood_heatmap_deg_baseline_vs_gvhd.pdf", width=4, height=10)

draw(hm, column_title_gp=gpar(fontsize=8, fontface="bold"), merge_legends=TRUE) 

dev.off()

pdf 
  2

# Clonality analysis

## Number of clonotypes per cell type over time

In [55]:
data <- so@meta.data %>% 
    dplyr::select(time_point, hto_demux_class, celltype_low, clone_id, clone_id_size) %>%
    dplyr::filter(celltype_low %in% grep("^T", names(color$celltype_low_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 
    dplyr::select(time_point, hto_demux_class, celltype_low, clone_id) %>% dplyr::distinct() %>% 
    dplyr::group_by(time_point, hto_demux_class, celltype_low) %>% dplyr::summarise(clonotype_n=n())

`summarise()` has grouped output by 'time_point', 'hto_demux_class'. You can
override using the `.groups` argument.


In [56]:
labels <- color$celltype_low_r
labels <- labels[names(labels) %in% levels(data$celltype_low)]

bp_1 <- ggplot(data, aes(x=time_point, y=clonotype_n, fill=celltype_low)) + 
    geom_bar(stat="identity", color="black", size=0.1) + 
    xlab("") + ylab("count") + 
    facet_grid(~hto_demux_class, scales="free") + 
    scale_fill_manual(values=color$celltype_low_r, labels=parse(text=names(labels))) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Cell type", reverse=FALSE)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

bp_2 <- ggplot(data, aes(x=time_point, y=log(clonotype_n), fill=celltype_low)) + 
    geom_bar(stat="identity", color="black", size=0.1) + 
    xlab("") + ylab("Clonotype log(count)") + 
    facet_grid(~hto_demux_class, scales="free") + 
    scale_fill_manual(values=color$celltype_low_r, labels=parse(text=names(labels))) + 
    guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title="Cell type", reverse=FALSE)) + 
    theme(axis.text.x=element_text(angle=90, vjust=0.5, hjust=1))

In [57]:
pdf("result/figures/figure_2/bp_tcell_clonotype_time_point_count.pdf", width=2.5, height=2)

gridExtra::grid.arrange(
    
    bp_1 %>% egg::set_panel_size(., width=unit(5*2.5, "mm"), height=unit(2.5, "cm"))

) 

dev.off()

pdf 
  2

In [58]:
pdf("result/figures/figure_2/bp_tcell_clonotype_time_point_log_count.pdf", width=2.5, height=2)

gridExtra::grid.arrange(
    
    bp_2 %>% egg::set_panel_size(., width=unit(5*2.5, "mm"), height=unit(2.5, "cm"))

) 

dev.off()

pdf 
  2

## Bar plots of clones per cell type over time points

In [59]:
data <- so@meta.data %>% 
    dplyr::select(time_point, hto_demux_class, celltype_low, clone_id, clone_id_size) %>% 
    dplyr::filter(celltype_low %in% grep("^T", names(color$celltype_low_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 
    dplyr::group_by(time_point, hto_demux_class, celltype_low, clone_id) %>% 
    dplyr::mutate(clone_id_size=n()) %>% 
    dplyr::ungroup() %>% dplyr::distinct() %>% 
    dplyr::mutate(clonotype_class=ifelse(clone_id_size == 1, "Singelton (N=1)", NA)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clone_id_size %in% 2:5, "Rare (N=2-5)", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=ifelse(clone_id_size %in% 6:20, "Intermediate (N=6-20)", clonotype_class)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clone_id_size > 20, "Expanded (N>20)", clonotype_class)) %>%
    dplyr::group_by(time_point, hto_demux_class, celltype_low) %>% 
    dplyr::mutate(clonotype_n=n()) %>% 
    dplyr::group_by(clonotype_class, clonotype_n, .add=TRUE) %>% 
    dplyr::summarise(clonotype_class_n=n()) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'clonotype_class'))) %>% 
    dplyr::ungroup() %>%
    dplyr::mutate(clonotype_class_ratio=clonotype_class_n/clonotype_n) %>% 
    dplyr::mutate(clonotype_class_ratio=ifelse(is.na(clonotype_class_ratio), 0, clonotype_class_ratio)) %>% 
    dplyr::mutate(clonotype_class=factor(clonotype_class, levels=rev(c("Singelton (N=1)", "Rare (N=2-5)", "Intermediate (N=6-20)", "Expanded (N>20)"))))

`summarise()` has grouped output by 'time_point', 'hto_demux_class',
'celltype_low', 'clonotype_class'. You can override using the `.groups`
argument.


In [60]:
data_1 <- dplyr::filter(data, hto_demux_class=="Blood") %>% droplevels()
data_2 <- dplyr::filter(data, hto_demux_class=="Skin") %>% droplevels()

In [61]:
bp_tcr <- function(data) {
    
    celltype_low_levels <- levels(data$celltype_low)
    
    data$celltype_low <- factor(data$celltype_low, levels=celltype_low_levels)
        
    custom_labeller <- function(x) {
                
        return(parse(text=x))
        
    }
    
    p <- ggplot(data, aes(x=time_point, y=clonotype_class_ratio, fill=clonotype_class)) +
        
            geom_bar(stat="identity", color="black", size=0.1) + 
            scale_fill_brewer(palette="Blues") + 
            xlab("") + ylab("Clonotype ratio") +
            scale_y_continuous(expand=c(0.01, 0.01)) + 
            facet_grid(~celltype_low, labeller=labeller(celltype_low=label_parsed)) +
    
            theme(

                axis.text.x=element_text(angle=90, vjust=0.5, hjust=1), 
                strip.text=element_text(size=8), 
                strip.background=element_rect(fill="transparent", color=NA)

                
            ) + 

            guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1), keywidth=0, keyheight=0.3, default.unit="cm", title="Clonality", reverse=FALSE)) 

    
    return(p)
    
}

In [62]:
p_1 <- bp_tcr(data_1)
p_2 <- bp_tcr(data_2)

In [63]:
pdf("result/figures/figure_2/bp_all_tcr_time_point_blood.pdf", width=1.5+0.5*length(levels(data_1$celltype_low)), height=2)

set_height <- 2.5
set_width <- rep(4, 16) * 2.5

gridExtra::grid.arrange(
    
    p_1 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), total_height=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

In [64]:
pdf("result/figures/figure_2/bp_all_tcr_time_point_skin.pdf", width=1.5+0.5*length(levels(data_2$celltype_low)), height=2)

set_height <- 2.5
set_width <- rep(4, 16) * 2.5

gridExtra::grid.arrange(
    
    p_2 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), total_height=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

## Clonotype species specificity 

In [65]:
data <- so@meta.data %>% 
    
    # Subset data 
    dplyr::select(genotype_class, time_point, hto_demux_class, celltype_low, clone_id, clone_id_size, species_vdjdb) %>% 
    dplyr::filter(celltype_low %in% grep("^T", names(color$celltype_low_r), value=TRUE)) %>% 
    na.omit() %>% droplevels()  

In [66]:
data <- so@meta.data %>% 
    
    # Subset data 
    dplyr::select(genotype_class, time_point, hto_demux_class, celltype_low, clone_id, clone_id_size, species_vdjdb) %>% 
    dplyr::filter(celltype_low %in% grep("^T", names(color$celltype_low_r), value=TRUE)) %>% 
    na.omit() %>% droplevels() %>% 

    # Set time_point 
    dplyr::mutate(time_point=ifelse(time_point=="GVHD", "GVHD", "HSCT")) %>% 
    dplyr::mutate(time_point=factor(time_point, levels=c("HSCT", "GVHD"))) %>% 
    
    # Recompute clonality for respective groups
    dplyr::group_by(hto_demux_class, time_point, clone_id) %>% dplyr::mutate(clonotype_n=n()) %>% 
    
    # Annotate clonality class
    dplyr::mutate(clonotype_class=ifelse(clonotype_n == 1, "Singelton", NA)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_n %in% 2:5, "Rare", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=ifelse(clonotype_n %in% 6:20, "Intermediate", clonotype_class)) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_n > 20, "Expanded", clonotype_class)) %>%
    dplyr::mutate(clonotype_class=factor(clonotype_class, levels=rev(c("Singelton", "Rare", "Intermediate", "Expanded")))) %>% 

    dplyr::ungroup() %>% 

    # Fix missing tcr annotations 
    dplyr::mutate(species_vdjdb=ifelse(species_vdjdb=="ambiguous", "Unknown", as.character(species_vdjdb))) %>% 

    dplyr::group_by(hto_demux_class, time_point, clone_id) %>%

    # Genotype class ratio 
    dplyr::group_by(clonotype_class, species_vdjdb, clonotype_n, .add=TRUE) %>% dplyr::mutate(clonotype_n=n()) %>% 
    dplyr::summarise(genotype_host_n=sum(genotype_class=="Host"), genotype_donor_n=sum(genotype_class=="Donor")) %>%
    dplyr::mutate(genotype_class=ifelse(genotype_host_n>0 & genotype_donor_n==0, "Host", NA)) %>% 
    dplyr::mutate(genotype_class=ifelse(genotype_host_n==0 & genotype_donor_n>0, "Donor", genotype_class)) %>% 
    dplyr::mutate(genotype_class=ifelse(genotype_host_n>0 & genotype_donor_n>0, "Shared", genotype_class)) %>% 
    
    # Compute ratios 
    dplyr::group_by(across(setdiff(group_vars(.), 'clone_id'))) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'species_vdjdb'))) %>% 
    dplyr::group_by(genotype_class, .add=TRUE) %>% dplyr::mutate(n=n()) %>% 
    dplyr::group_by(n, species_vdjdb, .add=TRUE) %>% dplyr::summarise(tcr_class_n=n()) %>% dplyr::ungroup() %>% 
    dplyr::mutate(tcr_class_ratio=tcr_class_n/n)

`summarise()` has grouped output by 'hto_demux_class', 'time_point',
'clone_id', 'clonotype_class', 'species_vdjdb'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'hto_demux_class', 'time_point',
'clonotype_class', 'genotype_class', 'n'. You can override using the `.groups`
argument.


In [67]:
species_vdjdb <- factor(data$species_vdjdb)
species_vdjdb <- relevel(species_vdjdb, "Unknown")
species_vdjdb <- levels(species_vdjdb)

In [68]:
color_species_vdjdb <- setNames(c("light gray", as.character(paletteer::paletteer_d("khroma::berlin"))[seq(1, 256, length.out=22) %>% ceiling()]), species_vdjdb)

In [69]:
piep <- function(data, title) {
    
    # Set species vdjdb factor levels to full level from color
    data$species_vdjdb <- factor(data$species_vdjdb, levels=names(color_species_vdjdb))
    data <- droplevels(data)
    
    p <- ggplot(data, aes(x=1, ymin=0, ymax=1, y=tcr_class_ratio, fill=species_vdjdb, xmin=0, xmax=1)) +
    
            geom_bar(stat="identity", color="black", size=0.1) + 
            coord_polar(theta="y", start=0) + 
            scale_fill_manual(values=color_species_vdjdb, drop=TRUE) + 
            ggtitle(title) + 
            facet_grid(clonotype_class~time_point) +

            theme(

                axis.text.x=element_blank(), 
                axis.text.y=element_blank(), 
                axis.title.x=element_blank(), 
                axis.title.y=element_blank(), 
                axis.ticks=element_blank(), 
                axis.line=element_blank()

            ) + 

            guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1), keywidth=0, keyheight=0.3, default.unit="cm", title=NULL, reverse=TRUE)) + 
            geom_text(aes(x=0, y=0, label=n), size=2, show.legend=FALSE)
        
    return(p)

}

In [70]:
piep_1 <- piep(data[data$hto_demux_class=="Blood" & data$genotype_class=="Host", ], title="Blood (Host)")
piep_2 <- piep(data[data$hto_demux_class=="Blood" & data$genotype_class=="Donor", ], title="Blood (Donor)")
piep_3 <- piep(data[data$hto_demux_class=="Skin" & data$genotype_class=="Host", ], title="Skin (Host)")
piep_4 <- piep(data[data$hto_demux_class=="Skin" & data$genotype_class=="Donor", ], title="Skin (Donor)")

In [71]:
pdf("result/figures/figure_2/pie_tcr_species_vdj_blood_host.pdf", width=4, height=3)

set_height <- rep(1, 3)
set_width <- rep(1, 2)

gridExtra::grid.arrange(
    
    piep_1 + ggh4x::force_panelsizes(cols=unit(set_width, "cm"), rows=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

In [72]:
pdf("result/figures/figure_2/pie_tcr_species_vdj_blood_donor.pdf", width=4, height=3)

set_height <- rep(1, 3)
set_width <- rep(1, 2)

gridExtra::grid.arrange(
    
    piep_2 + ggh4x::force_panelsizes(cols=unit(set_width, "cm"), rows=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

In [73]:
pdf("result/figures/figure_2/pie_tcr_species_vdj_skin_host.pdf", width=4, height=3)

set_height <- rep(1, 3)
set_width <- rep(1, 2)

gridExtra::grid.arrange(
    
    piep_3 + ggh4x::force_panelsizes(cols=unit(set_width, "cm"), rows=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

In [74]:
pdf("result/figures/figure_2/pie_tcr_species_vdj_skin_donor.pdf", width=4, height=3)

set_height <- rep(1, 3)
set_width <- rep(1, 2)

gridExtra::grid.arrange(
    
    piep_4 + ggh4x::force_panelsizes(cols=unit(set_width, "cm"), rows=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

## Donut plot shared clones between blood and skin per patient id

In [172]:
data <- so@meta.data %>% 
    dplyr::filter(celltype_low %in% grep("^T", names(color$celltype_low_r), value=TRUE)) %>% 
    # dplyr::filter(time_point %in% c("D14", "D100", "GVHD")) %>% 
    droplevels() %>% 
    dplyr::select(patient_id, clone_id, hto_demux_class) %>% na.omit() %>% dplyr::distinct() %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::summarise(clonotype_class=n()) %>% 
    dplyr::mutate(clonotype_class=ifelse(clonotype_class==1, "private", "shared")) %>% 
    dplyr::group_by(across(setdiff(group_vars(.), 'clone_id'))) %>% 
    dplyr::mutate(clonotype_n=n()) %>%
    dplyr::group_by(clonotype_class, clonotype_n, .add=TRUE) %>% dplyr::summarise(clonotype_class_n=n()) %>% dplyr::ungroup() %>% 
    tidyr::complete(., patient_id, clonotype_class, fill=list(clonotype_class_n=0), explicit=TRUE) %>% 
    dplyr::mutate(clonotype_class_ratio=clonotype_class_n/clonotype_n) %>% 
    dplyr::mutate(clonotype_class_ratio=ifelse(is.na(clonotype_class_ratio), 0, clonotype_class_ratio)) %>% 
    dplyr::mutate(patient_id=gsub("Patient_", "P", patient_id)) %>% 
    dplyr::mutate(patient_id=factor(patient_id, levels=paste0("P", 1:12))) %>% droplevels()

`summarise()` has grouped output by 'patient_id'. You can override using the
`.groups` argument.
`summarise()` has grouped output by 'patient_id', 'clonotype_class'. You can
override using the `.groups` argument.


In [76]:
piep_1 <- ggplot(data, aes(x=2, y=clonotype_class_ratio, fill=clonotype_class)) +
        
        geom_bar(stat="identity", color="white") + 
        
        coord_polar(theta="y", start=0) + 
        scale_fill_brewer(palette="Blues") + 
        
        facet_wrap(~patient_id, nrow=2) +
                
        theme(
        
            axis.text.x=element_blank(), 
            axis.text.y=element_blank(), 
            axis.title.x=element_blank(), 
            axis.title.y=element_blank(), 
            axis.ticks=element_blank(), 
            axis.line=element_blank()
        
        ) +

        guides(fill=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.1, default.unit="cm", title=NULL, reverse=TRUE)) + 

        geomtextpath::geom_textpath(
            data=data, 
            position=position_stack(vjust=0.25),
            aes(x=2, y=clonotype_class_ratio, label=paste0("N=", clonotype_class_n)),
            size=2, 
            show.legend=FALSE
        ) 

In [77]:
pdf("result/figures/figure_2/pie_shared_tcr_patient_id.pdf", width=5, height=3)

set_height <- rep(1.5, 2)
set_width <- rep(1.5, 6)

gridExtra::grid.arrange(
    
    piep_1 + ggh4x::force_panelsizes(cols=unit(set_width, "cm"), rows=unit(set_height, "cm"))

)

dev.off()

pdf 
  2

## Flow plot of blood and skin shared cell types

In [78]:
data <- so@meta.data %>% 
    dplyr::filter(celltype_low %in% (grep("^T", names(color$celltype_low_r), value=TRUE) %>% setdiff('T[\'inv\']'))) %>% 
    droplevels() %>% 
    dplyr::select(cell_id, patient_id, clone_id, hto_demux_class, celltype_low, time_point) %>% na.omit() %>% 
    dplyr::group_by(patient_id, clone_id) %>% 
    dplyr::mutate(clonotype_class=ifelse(length(unique(hto_demux_class))==1, "private", "shared")) %>% 
    dplyr::filter(clonotype_class=="shared") %>% dplyr::ungroup() %>% 
    dplyr::mutate(time_point=ifelse(time_point %in% c("D14", "D100"), "HSCT", as.character(time_point))) %>% droplevels() %>% dplyr::ungroup()

In [79]:
data %>% dplyr::select(-hto_demux_class, -clonotype_class, -celltype_low) %>% dplyr::distinct() %>% dplyr::mutate(clone_id=paste0("id_", clone_id)) %>% 
    dplyr::group_by(patient_id, clone_id) %>% dplyr::summarise(clone_cells_n=n()) %>% dplyr::ungroup() %>% 
    dplyr::group_by(patient_id) %>% dplyr::mutate(clone_n=n()) %>% dplyr::ungroup() %>% 
    dplyr::arrange(patient_id) %>% write.csv("result/ir/ir_shared_blood_skin_counts.csv")

`summarise()` has grouped output by 'patient_id'. You can override using the
`.groups` argument.


In [80]:
data_1 <- data %>%
    droplevels() %>% 
    dplyr::select(clone_id, hto_demux_class, celltype_low, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(clone_id, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(celltype_low))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(clone_id) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(celltype_low=paste0(celltype_low, "_", hto_demux_class)) %>% 
    reshape2::dcast(., clone_id~hto_demux_class, value.var="celltype_low") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [81]:
data_2 <- data %>%
    dplyr::filter(time_point=="Baseline") %>% droplevels() %>% 
    dplyr::select(clone_id, hto_demux_class, celltype_low, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(clone_id, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(celltype_low))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(clone_id) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(celltype_low=paste0(celltype_low, "_", hto_demux_class)) %>% 
    reshape2::dcast(., clone_id~hto_demux_class, value.var="celltype_low") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [82]:
data_3 <- data %>%
    dplyr::filter(time_point=="HSCT") %>% droplevels() %>% 
    dplyr::select(clone_id, hto_demux_class, celltype_low, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(clone_id, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(celltype_low))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(clone_id) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(celltype_low=paste0(celltype_low, "_", hto_demux_class)) %>% 
    reshape2::dcast(., clone_id~hto_demux_class, value.var="celltype_low") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [83]:
data_4 <- data %>%
    dplyr::filter(time_point=="GVHD") %>% droplevels() %>% 
    dplyr::select(clone_id, hto_demux_class, celltype_low, time_point) %>% dplyr::distinct() %>% 
    dplyr::group_by(clone_id, hto_demux_class, time_point) %>% dplyr::mutate(n=length(unique(celltype_low))) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==1) %>% 
    dplyr::group_by(clone_id) %>% dplyr::mutate(n=n()) %>% dplyr::ungroup() %>% 
    dplyr::filter(n==2) %>% 
    dplyr::mutate(celltype_low=paste0(celltype_low, "_", hto_demux_class)) %>% 
    reshape2::dcast(., clone_id~hto_demux_class, value.var="celltype_low") %>% 
    group_by(Blood, Skin) %>% summarise(n=n()) %>% tidyr::spread(key=Skin, value=n, fill=0) %>% 
    tibble::column_to_rownames("Blood") %>% as.matrix()

`summarise()` has grouped output by 'Blood'. You can override using the
`.groups` argument.


In [84]:
chord_diagram <- function(data) {
    
    # Suppress the plot display
    pdf(NULL)
    dev.control(displaylist="enable")
    
    # Set grid colors
    color_1 <- color$celltype_low_r
    color_2 <- color$celltype_low_r

    names(color_1) <- paste0(names(color_1), "_Blood")
    names(color_2) <- paste0(names(color_2), "_Skin")

    grid_col <- c(color_1, color_2)
    grid_col <- grid_col[names(grid_col) %in% c(colnames(data), rownames(data))]

    # Set sector index labels
    labels <- setNames(gsub("_Blood|_Skin", "", names(grid_col)), names(grid_col))

    # Create the chord diagram
    chordDiagram(data, grid.col=grid_col, annotationTrack=c("grid"), annotationTrackHeight=mm_h(10))
    
    abline(h=0, lty=2, col="#000000")
    
    # Set Axis 
    for(sector_index in get.all.sector.index()) {
        
        circos.xaxis(sector.index=sector_index, minor.ticks=0, labels.cex=0.25, labels=FALSE)
        
    }
    
    # Set labels 
    circos.trackPlotRegion(track.index=1, panel.fun=function(x, y) {
        
        # Get sector labels and replace 
        sector.name <- get.cell.meta.data("sector.index")
        labels <- labels[match(sector.name, names(labels))]
        
        circos.text(x=CELL_META$xcenter, y=CELL_META$ylim[2], labels=parse(text=labels), cex=0.5, niceFacing=TRUE, facing="outside", adj=c(0.5, 1.5))

    }, bg.border=NA
                          )
    
    # Record plot
    p <- recordPlot()
    
    # Close the null device
    invisible(dev.off())
    
    return(p)
    
}

In [85]:
plot_1 <- chord_diagram(data_1)
plot_2 <- chord_diagram(data_2)
plot_3 <- chord_diagram(data_3)
plot_4 <- chord_diagram(data_4)

In [86]:
pdf("result/figures/figure_2/choard_diagram_shared_tcr_all.pdf", width=2, height=2)

replayPlot(plot_1)

dev.off()

pdf 
  2

In [87]:
pdf("result/figures/figure_2/choard_diagram_shared_tcr_baseline.pdf", width=2, height=2)

replayPlot(plot_2)

dev.off()

pdf 
  2

In [88]:
pdf("result/figures/figure_2/choard_diagram_shared_tcr_hsct.pdf", width=2, height=2)

replayPlot(plot_3)

dev.off()

pdf 
  2

In [89]:
pdf("result/figures/figure_2/choard_diagram_shared_tcr_gvhd.pdf", width=2, height=2)

replayPlot(plot_4)

dev.off()

pdf 
  2

## Skin vs Blood TCR clone DEA 

In [90]:
feature_select <- function(so, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE) {
    
    # Select Seurat components 
    meta <- so@meta.data
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    
    # Prepare data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
    
    # Split expression matrix by grouping variable into list
    cnt <- split(cnt, f=paste(so[[grouping_var, drop=TRUE]], so[[random_effect_var, drop=TRUE]]))
    
    # Check within group expression 
    cnt <- lapply(cnt, function(x) {colSums(x>=cnt_min)>=cell_min})
    cnt <- do.call(rbind, cnt)
    
    # Check across group expression 
    cnt_check <- colSums(cnt)>=feature_group_min
    
    # Get genes 
    genes <- colnames(cnt)[cnt_check]

    # Create Seurat object with gene subset 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt <- cnt[genes, , drop=FALSE]

    so <- CreateSeuratObject(counts=cnt, meta=meta)
    
    if(verbose) {message(paste("Final number of genes for testing", nrow(so)))}

    return(so)
    
}

In [91]:
voom_lm_fit <- function(so, grouping_var, random_effect_var, sample_weights) {

    # Get Counts 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
        
    # Prepare count data for split 
    cnt <- t(as.matrix(cnt))
    cnt <- as.data.frame(cnt)
        
    # Make pseudobulks by suming single cell counts
    cnt <- split(cnt, f=paste0(so[[grouping_var, drop=TRUE]], "|", so[[random_effect_var, drop=TRUE]]))
    cnt <- lapply(names(cnt), function(i) {x <- data.frame(counts=colSums(cnt[[i]])); colnames(x) <- i; return(x)})
    cnt <- do.call(cbind, cnt)
        
    # Get grouping variables from cnt matrix
    grouping_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 1) # Used for design matrix
    random_effect_var_vec <- sapply(strsplit(colnames(cnt), "\\|"), `[[`, 2) # Used as blocking variable
    
    # Design matrix 
    grouping_var_vec <- as.character(grouping_var_vec)
    design <- model.matrix(~0+grouping_var_vec)
    colnames(design) <- gsub("grouping_var_vec", "", colnames(design))
    
    # Run voomLmFit
    fit <- edgeR::voomLmFit(
        
        counts=cnt, 
        design=design, 
        block=random_effect_var_vec, 
        normalize.method="none", 
        lib.size=NULL, 
        sample.weights=sample_weights, 
        plot=FALSE

    )
    
    return(fit)
    
}

In [92]:
ebayes <- function(fit, so, grouping_var, grouping_var_vec) {
    
    contrasts_vec <- paste0(grouping_var_vec[1], "-", grouping_var_vec[2])
    contrasts <- limma::makeContrasts(contrasts=contrasts_vec, levels=fit$design)
    contrasts_fit <- limma::contrasts.fit(fit, contrasts=contrasts)
    
    # eBayes fit 
    efit <- limma::eBayes(contrasts_fit)

    # Get result table
    result_df <- limma::topTable(efit, sort.by="P", n=Inf, p.value=1, lfc=0, coef=1)
    
    # Convert results to Seurat format 
    colnames(result_df)[1] <- "avg_log2FC"
    colnames(result_df)[4] <- "p_value"
    colnames(result_df)[5] <- "p_val_adj"

    # Adjusted p-values with IHW
    result_df$p_val_adj <- IHW::adj_pvalues(IHW::ihw(result_df$p_value ~ result_df$AveExpr, alpha=0.05))
            
    # Add percentage expression 
    cnt <- GetAssayData(so, assay="RNA", slot="counts")
    cnt_bool <- cnt[rownames(result_df), ] > 0

    result_df$pct_1<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[1]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[1])
    result_df$pct_2<-rowSums(cnt_bool[, so[[grouping_var, drop=TRUE]]==grouping_var_vec[2]])/sum(so[[grouping_var, drop=TRUE]]==grouping_var_vec[2])
    
    return(result_df)
    
}

In [93]:
grouping_var <- "hto_demux_class"
random_effect_var <- "patient_id"
grouping_var_vec <- c("Skin", "Blood")
sample_weights <- FALSE

cnt_min=3
cell_min=3
feature_group_min=1

p_adj_thr <- 0.05

so_1 <- subset(so, subset=clone_id %in% data[data$time_point=="Baseline", ]$clone_id)
so_2 <- subset(so, subset=clone_id %in% data[data$time_point=="HSCT", ]$clone_id)
so_3 <- subset(so, subset=clone_id %in% data[data$time_point=="GVHD", ]$clone_id)

In [94]:
so_1 <- feature_select(so_1, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)
so_2 <- feature_select(so_2, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)
so_3 <- feature_select(so_3, cnt_min, cell_min, grouping_var, random_effect_var, feature_group_min, verbose=TRUE)

Final number of genes for testing 1829

Final number of genes for testing 1428

Final number of genes for testing 1436



In [95]:
fit_1 <- voom_lm_fit(so_1, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)
fit_2 <- voom_lm_fit(so_2, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)
fit_3 <- voom_lm_fit(so_3, grouping_var=grouping_var, random_effect_var=random_effect_var, sample_weights=sample_weights)

First intra-block correlation  0.4037028

Final intra-block correlation  0.4046772

First intra-block correlation  0.3409647

Final intra-block correlation  0.3416558

First intra-block correlation  0.3562734

Final intra-block correlation  0.352307



In [96]:
result_1 <- ebayes(fit_1, so_1, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)
result_2 <- ebayes(fit_2, so_2, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)
result_3 <- ebayes(fit_3, so_3, grouping_var=grouping_var, grouping_var_vec=grouping_var_vec)

Only 1 bin; IHW reduces to Benjamini Hochberg (uniform weights)

Only 1 bin; IHW reduces to Benjamini Hochberg (uniform weights)

Only 1 bin; IHW reduces to Benjamini Hochberg (uniform weights)



In [97]:
openxlsx::write.xlsx(list("Baseline"=result_1, "HSCT"=result_2, "GVHD"=result_3), "result/dea/pseudobulk/tcr/shared_clones_tissue.xlsx", colNames=TRUE, rowNames=TRUE)

In [98]:
# Label genes 
label <- read.csv("script/figure/figure_2/data/figure_2_shared_tcr_clonotypes_dea.csv")

In [99]:
vp_1 <- dea_vp(result_1, title="Blood vs Skin TCR clones (Baseline)", color_neg=color$hto_demux_class["Blood"], color_pos=color$hto_demux_class["Skin"], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=20, parse_title=FALSE, label=label[["gene"]])
vp_2 <- dea_vp(result_2, title="Blood vs Skin TCR clones (HSCT)", color_neg=color$hto_demux_class["Blood"], color_pos=color$hto_demux_class["Skin"], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=20, parse_title=FALSE, label=label[label$group=="HSCT", ][["gene"]])
vp_3 <- dea_vp(result_3, title="Blood vs Skin TCR clones (GVHD)", color_neg=color$hto_demux_class["Blood"], color_pos=color$hto_demux_class["Skin"], log2fc_thr=0, p_adj_thr=p_adj_thr, top_label=20, parse_title=FALSE, label=label[label$group=="GVHD", ][["gene"]])

In [100]:
pdf("result/figures/figure_2/vp_dea_shared_tcr.pdf", width=6, height=2)

gridExtra::grid.arrange(
    
    vp_1 %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_2 %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")),
    vp_3 %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), ncol=3

) 

dev.off()

pdf 
  2

# DEA overview T cells

## Skin 

In [101]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_skin.rds")
grouping_stat <- readRDS("result/dea/pseudobulk/time_point/celltype_low/grouping_stat_skin.rds")

In [102]:
cell_type_order <- levels(so$celltype_low)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [103]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [104]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cell count mean", name="log2(mean cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID count", name="Count (Patient ID)", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG count", name="Count (DEG)", row_order=cell_type_order)

ERROR: Error in grouping_stat_hm(grouping_stat, stat = "cell_mean", column_title = "Cell count mean", : could not find function "grouping_stat_hm"


In [ ]:
pdf("result/figures/figure_2/hm_dea_stat_skin_t_cells.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

## Blood 

In [ ]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_blood.rds")
grouping_stat <- readRDS("result/dea/pseudobulk/time_point/celltype_low/grouping_stat_blood.rds")

In [ ]:
cell_type_order <- levels(so$celltype_low)
cell_type_order <- cell_type_order[cell_type_order %in% names(results)]
p_adj_thr <- 0.05

In [ ]:
results <- results[names(results) %in% cell_type_order]
grouping_stat <- grouping_stat[names(results) %in% cell_type_order]

In [ ]:
source("script/figure/bin/dea_stats.R")

hm_1 <- grouping_stat_hm(grouping_stat, stat="cell_mean", column_title="Cell count mean", name="log2(mean cells)", row_order=cell_type_order)
hm_2 <- grouping_stat_hm(grouping_stat, stat="random_effect_var_n", column_title="Patient ID count", name="Count (Patient ID)", row_order=cell_type_order)
hm_3 <- results_n_hm(results, p_adj_thr=p_adj_thr, column_title="DEG count", name="Count (DEG)", row_order=cell_type_order)

In [ ]:
pdf("result/figures/figure_2/hm_dea_stat_blood_t_cells.pdf", width=3, height=3)

draw(hm_1 + hm_2 + hm_3, heatmap_legend_side="bottom")

dev.off()

# DEA volcano plot 

## Results for pseudobatch lymphocyte blood (PB)

In [ ]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_blood.rds")
results <- results[levels(so$celltype_low)]

In [ ]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [ ]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [ ]:
pdf("result/figures/figure_2/vp_dea_pb_t_cells_blood.pdf", width=8, height=4)

gridExtra::grid.arrange(

    vp_list[[1]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[2]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")),
    vp_list[[3]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[4]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")),
    vp_list[[5]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[6]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[7]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), ncol=4

) 

dev.off()

## Results for pseudobatch lymphocyte skin (PB)

In [ ]:
results <- readRDS("result/dea/pseudobulk/time_point/celltype_low/results_skin.rds")
results <- results[levels(so$celltype_low)]

In [ ]:
results <- lapply(results, function(result) result[["GVHD-Baseline"]])
results <- results[!sapply(results, is.null)]

In [ ]:
vp_list <- lapply(names(results), function(i) dea_vp(results[[i]], title=i, color_neg=color$time_point["Baseline"], color_pos=color$time_point["GVHD"]))

In [ ]:
pdf("result/figures/figure_2/vp_dea_pb_t_cells_skin.pdf", width=8, height=2)

gridExtra::grid.arrange(

    vp_list[[1]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[2]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")),
    vp_list[[3]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), 
    vp_list[[4]] %>% egg::set_panel_size(., width=unit(3, "cm"), height=unit(3, "cm")), ncol=4

) 

dev.off()

# Residency score 

In [107]:
ms <- read.csv("result/ms/ms.csv", row.names=1)
so <- AddMetaData(so, ms)

## TRM score

In [ ]:
data <- so@meta.data

In [ ]:
pdf("result/figures/figure_2/jvp_trm_score_cd4pos.pdf", width=5, height=2)

p_1 <- ggplot(data[data$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'"), ], aes(x=genotype_class, y=TRM_ucell, color=genotype_class)) + 
    
    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_boxplot(width=0.5, size=0.25, outliers=FALSE, alpha=0, color="black") +     
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD4+") + xlab("") + ylab("Module score diff.") + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

set_height <- 2.5
set_width <- rep(2, 10) * 2.5

gridExtra::grid.arrange(
    
    p_1 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), rows=unit(set_height, "cm"))

)

dev.off()

In [ ]:
pdf("result/figures/figure_2/jvp_trm_score_cd8pos.pdf", width=5, height=2)

p_2 <- ggplot(data[data$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'"), ], aes(x=genotype_class, y=TRM_ucell, color=genotype_class)) +

    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_boxplot(width=0.5, size=0.25, outliers=FALSE, alpha=0, color="black") +    
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD8+") + xlab("") + ylab("Module score diff.") + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

set_height <- 2.5
set_width <- rep(2, 10) * 2.5

gridExtra::grid.arrange(
    
    p_2 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), rows=unit(set_height, "cm"))

)

dev.off()

In [165]:
sc_score_wilcox <- function(so, ident, ident_1=NULL, ident_2=NULL, only_pos=FALSE, avg_log2FC_threshold=0, pct_min=0, cnt_min=0, cell_min=0, test_use="wilcox", verbose=TRUE) {

    score <- so@meta.data[, grep("ucell", colnames(so@meta.data), value=TRUE), drop=FALSE]
    so[["UCELL"]] <- CreateAssayObject(data=t(score), check.matrix=TRUE)
    
    # Check number of cells 
    n_cells_1 <- sum(so@meta.data[[ident]]==ident_1)
    n_cells_2 <- sum(so@meta.data[[ident]]==ident_2)

    check_1 <- n_cells_1 >= 3
    check_2 <- n_cells_2 >= 3
    
    if(check_1 & check_2) {
        
        so <- SetIdent(so, value=ident)
        res <- RunPresto(so, ident.1=ident_1, ident.2=ident_2, logfc.threshold=avg_log2FC_threshold, min.pct=0, only.pos=only_pos, test.use=test_use, assay="UCELL", layer="data")

        # Annotate results 
        res$score <- rownames(res)

        # N cells per group 
        res$n_cells_1 <- n_cells_1
        res$n_cells_2 <- n_cells_2

        return(res)
        
    } else {
        
        return(NULL)
        
    }

}

In [166]:
res <- list(
    
    "Blood_D100"=sc_score_wilcox(so[, so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$hto_demux_class=="Blood" & so$time_point=="D100"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Blood_GVHD"=sc_score_wilcox(so[, so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$hto_demux_class=="Blood" & so$time_point=="GVHD"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Skin_D100"=sc_score_wilcox(so[, so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$hto_demux_class=="Skin" & so$time_point=="D100"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Skin_GVHD"=sc_score_wilcox(so[, so$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'") & so$hto_demux_class=="Skin" & so$time_point=="GVHD"], "genotype_class", ident_1="Host", ident_2="Donor")

)

In [167]:
openxlsx::write.xlsx(res, "result/statistics/th_genotype_ucell_score_wilcox.xlsx", colNames=TRUE, rowNames=TRUE)

## TCM score

In [ ]:
data <- so@meta.data

In [ ]:
pdf("result/figures/figure_2/jvp_tcm_score_cd4pos.pdf", width=5, height=2)

p_1 <- ggplot(data[data$celltype_low %in% c("T['h']*' N/CM'", "T['h']*' E/EM'", "T['h']*' RM'"), ], aes(x=genotype_class, y=TCM_ucell, color=genotype_class)) + 
    
    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_boxplot(width=0.5, size=0.25, outliers=FALSE, alpha=0, color="black") +     
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD4+") + xlab("") + ylab("Module score diff.") + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

set_height <- 2.5
set_width <- rep(2, 10) * 2.5

gridExtra::grid.arrange(
    
    p_1 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), rows=unit(set_height, "cm"))

)

dev.off()

In [ ]:
pdf("result/figures/figure_2/jvp_tcm_score_cd8pos.pdf", width=5, height=2)

p_2 <- ggplot(data[data$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'"), ], aes(x=genotype_class, y=TCM_ucell, color=genotype_class)) +

    geom_jitter(shape=19, size=0.1, width=0.25) + 
    geom_boxplot(width=0.5, size=0.25, outliers=FALSE, alpha=0, color="black") +    
    geom_hline(yintercept=0, linetype="dotted", colour="black") + 
    ggtitle("T cell CD8+") + xlab("") + ylab("Module score diff.") + 
    scale_color_manual(values=color$genotype_class) + 
    facet_grid(~hto_demux_class+time_point) + 
    theme(axis.text.x=element_blank()) + 
    guides(color=guide_legend(ncol=1, override.aes=list(alpha=1, size=1.5), keywidth=0, keyheight=0.25, default.unit="cm"))

set_height <- 2.5
set_width <- rep(2, 10) * 2.5

gridExtra::grid.arrange(
    
    p_2 + ggh4x::force_panelsizes(cols=unit(set_width, "mm"), rows=unit(set_height, "cm"))

)

dev.off()

In [168]:
res <- list(
    
    "Blood_D100"=sc_score_wilcox(so[, so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$hto_demux_class=="Blood" & so$time_point=="D100"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Blood_GVHD"=sc_score_wilcox(so[, so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$hto_demux_class=="Blood" & so$time_point=="GVHD"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Skin_D100"=sc_score_wilcox(so[, so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$hto_demux_class=="Skin" & so$time_point=="D100"], "genotype_class", ident_1="Host", ident_2="Donor"), 
    "Skin_GVHD"=sc_score_wilcox(so[, so$celltype_low %in% c("T['c']*' N/CM'", "T['c']*' E/EM'", "T['c']*' RM'") & so$hto_demux_class=="Skin" & so$time_point=="GVHD"], "genotype_class", ident_1="Host", ident_2="Donor")

)

In [169]:
openxlsx::write.xlsx(res, "result/statistics/tc_genotype_ucell_score_wilcox.xlsx", colNames=TRUE, rowNames=TRUE)